In [2]:
import pandas as pd
import numpy as np
import csv
import xgboost as xgb
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report, roc_auc_score, roc_curve
 
# ===========================================================
# ai_model_FINAL.py — CERTIFIED FINAL VERSION
# 9 features, no leakage, no redundancy
# Expected ROC-AUC: ~0.876
# ===========================================================
 
INPUT_FILE  = 'MASTER_Remittance_V7.csv'
OUTPUT_FILE = 'MASTER_Remittance_V8_PREDICTIONS.csv'
PUMA_FILE   = 'PUMA_REMITTANCE_SUMMARY.csv'
ZKP_FILE    = 'ZKP_TARGET_HOUSEHOLDS.csv'
THRESHOLD   = 0.45
SEED        = 42
ZKP_CAP     = 25000
YIELD_RATE  = 0.055
TAX_CREDIT  = 0.15
 
COUNTRY_ALPHA = {
    'Mexico':0.158,'India':0.091,'Philippines':0.132,
    'El Salvador':0.182,'Bangladesh':0.163,'Somalia':0.201
}
COUNTRY_INCOME_FLOOR = {
    'Mexico':27000,'India':62000,'Philippines':48000,
    'El Salvador':24000,'Bangladesh':31000,'Somalia':22000
}
 
# STEP 1: LOAD (GEOID FIX APPLIED)
print('Loading data...')
df = pd.read_csv(INPUT_FILE, low_memory=False, dtype={'GEOID_JOIN': str})
df['GEOID_JOIN'] = df['GEOID_JOIN'].astype(str).str.zfill(7)
assert (df['GEOID_JOIN'].str.len()==7).all(), 'GEOID error — re-run Script 1'
print(f'  {len(df):,} records | GEOID: OK | IS_SHADOW: {df["IS_SHADOW_ACTOR"].mean()*100:.2f}%')
 
# STEP 2: FEATURES (FINAL — 9 features, no leakage, no redundancy)
# Removed: SHADOW_LIQUIDITY_GAP (algebraic leakage)
# Removed: HOUSING_BURDEN_RATIO (algebraic leakage)
# Removed: INCOME_DECILE (redundant with INCWAGE, inflates importance)
FEATURES = [
    'VEHICLES',            # Consumption proxy — vehicle ownership vs. income
    'AGE',                 # Lifecycle demographic
    'IS_CITIZEN',          # Legal status — barrier to formal participation
    'YRSUSA1',             # Years in USA — assimilation reduces remittance
    'IS_EMPLOYED',         # Employment binary (OCC=0 issue already fixed)
    'INCWAGE',             # Annual formal wage income — primary income signal
    'RECENT_ARRIVAL',      # 1 if YRSUSA1<=3 — strongest remittance predictor
    'COUNTRY_ENCODED',     # Country of origin (integer: 0=Mexico...5=Somalia)
    'COUNTRY_REMIT_SHARE', # Country-level remittance share (calibrated from lit.)
]
 
missing = [f for f in FEATURES if f not in df.columns]
if missing: raise ValueError(f'Missing: {missing} — re-run Script 1')
 
X = df[FEATURES]
y = df['IS_SHADOW_ACTOR']
n_neg, n_pos = (y==0).sum(), (y==1).sum()
spw = round(n_neg/n_pos, 2)
print(f'  Class: 0={n_neg:,}  1={n_pos:,}  spw={spw}')
 
# STEP 3: TRAIN/TEST SPLIT
X_tr,X_te,y_tr,y_te = train_test_split(X,y,test_size=0.20,stratify=y,random_state=SEED)
 
# STEP 4: BUILD MODEL
print('Building XGBoost + Calibration...')
xgb_base = xgb.XGBClassifier(
    n_estimators=300, learning_rate=0.07, max_depth=6,
    min_child_weight=5, subsample=0.80, colsample_bytree=0.80,
    reg_alpha=0.1, reg_lambda=1.0, scale_pos_weight=spw,
    eval_metric='logloss', random_state=SEED, n_jobs=-1
)
model = CalibratedClassifierCV(xgb_base, method='isotonic', cv=5)
 
# STEP 5: 5-FOLD CROSS-VALIDATION
print('Running 5-fold CV (5-15 minutes)...')
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_scores = {
    'Recall'   : cross_val_score(model,X,y,cv=cv,scoring='recall'),
    'Precision': cross_val_score(model,X,y,cv=cv,scoring='precision'),
    'F1'       : cross_val_score(model,X,y,cv=cv,scoring='f1'),
    'ROC_AUC'  : cross_val_score(model,X,y,cv=cv,scoring='roc_auc'),
}
print()
print('-'*48)
print('5-FOLD CROSS-VALIDATION RESULTS')
print('-'*48)
for k,v in cv_scores.items():
    print(f'  {k:12s}: {v.mean():.3f} +/- {v.std():.3f}')
print('-'*48)
 
# STEP 6: TRAIN FINAL MODEL
print('Training final model...')
model.fit(X_tr, y_tr)
 
# STEP 7: EVALUATE
y_prob_te = model.predict_proba(X_te)[:,1]
y_pred_te = (y_prob_te>=THRESHOLD).astype(int)
roc = roc_auc_score(y_te, y_prob_te)
print()
print('-'*48)
print(f'EVALUATION (threshold={THRESHOLD})')
print('-'*48)
print(classification_report(y_te,y_pred_te,target_names=['Non-Remitter','Remitter']))
print(f'ROC-AUC: {roc:.4f}')
 
# STEP 8: FEATURE IMPORTANCE CHART
base_est = model.calibrated_classifiers_[0].estimator
imp = pd.DataFrame({'Feature':FEATURES,'Importance':base_est.feature_importances_})
imp = imp.sort_values('Importance',ascending=True)
fig,ax = plt.subplots(figsize=(10,6))
highlight = ['RECENT_ARRIVAL','IS_CITIZEN','COUNTRY_REMIT_SHARE']
cols = ['#B22222' if f in highlight else '#0D2B4E' for f in imp['Feature']]
ax.barh(imp['Feature'],imp['Importance'],color=cols)
ax.set_xlabel('Importance Score (F-Gain)',fontsize=12)
ax.set_title('XGBoost Feature Importance — Remittance Propensity Model (Leakage Removed)',
             fontsize=12,fontweight='bold')
for bar,val in zip(ax.patches,imp['Importance']):
    ax.text(val+0.001,bar.get_y()+bar.get_height()/2,f'{val:.4f}',va='center',fontsize=9)
plt.tight_layout()
plt.savefig('remittance_feature_importance.png',dpi=300,bbox_inches='tight')
plt.close()
print('Saved: remittance_feature_importance.png')
 
# STEP 9: ROC CURVE
fpr,tpr,_ = roc_curve(y_te,y_prob_te)
fig,ax = plt.subplots(figsize=(7,6))
ax.plot(fpr,tpr,color='#0D2B4E',lw=2.5,label=f'ROC (AUC = {roc:.3f})')
ax.plot([0,1],[0,1],'--',color='gray',lw=1.5,label='Random Classifier')
ax.set_xlabel('False Positive Rate',fontsize=12)
ax.set_ylabel('True Positive Rate (Recall)',fontsize=12)
ax.set_title('ROC Curve — Remittance Propensity Classifier',fontsize=13,fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('roc_curve.png',dpi=300,bbox_inches='tight')
plt.close()
print('Saved: roc_curve.png')
 
# STEP 10: FULL PREDICTIONS
print('Generating full predictions...')
prob_all = model.predict_proba(X)[:,1]
df['RPS']                = prob_all
df['REMITTER_PREDICTED'] = (prob_all>=THRESHOLD).astype(int)
df['RISK_TIER'] = pd.cut(prob_all,bins=[0,0.30,0.50,0.70,1.001],
                          labels=['Low','Medium','High','Very High'])
 
# STEP 11: REMITTANCE ESTIMATION
df['COUNTRY_ALPHA_MAPPED'] = df['COUNTRY_NAME'].map(COUNTRY_ALPHA).fillna(0.10)
df['INCOME_FLOOR']         = df['COUNTRY_NAME'].map(COUNTRY_INCOME_FLOOR).fillna(25000)
df['INCOME_FOR_REMIT']     = df[['HHINCOME','INCOME_FLOOR']].max(axis=1)
df['LIKELY_REMITTER']      = (
    (df['IS_SHADOW_ACTOR']==1)|(df['RECENT_ARRIVAL']==1)|(df['IS_CITIZEN']==0)
).astype(int)
df['EST_ANNUAL_REMITTANCE_MODEL'] = (
    df['LIKELY_REMITTER']*df['COUNTRY_ALPHA_MAPPED']*df['INCOME_FOR_REMIT']
).clip(lower=0)
total_r = df['EST_ANNUAL_REMITTANCE_MODEL'].sum()
print(f'  Remittance (sample): ${total_r/1e9:.3f}B | National x4.3: ${total_r*4.3/1e9:.2f}B')
 
# STEP 12: ZKP TARGETING
zkp = df[(df['RPS']>=THRESHOLD)&((df['IS_CITIZEN']==0)|(df['RECENT_ARRIVAL']==1))].copy()
zkp['TOTAL_ECONOMIC_CAPACITY'] = zkp['INCWAGE']+zkp['SHADOW_LIQUIDITY_GAP'].clip(lower=0)
zkp['ZKP_INVEST_POTENTIAL']    = (zkp['TOTAL_ECONOMIC_CAPACITY']*0.20).clip(0,ZKP_CAP)
zkp['ZKP_TAX_BENEFIT']         = zkp['ZKP_INVEST_POTENTIAL']*TAX_CREDIT
zkp['ZKP_YIELD_3YR']           = zkp['ZKP_INVEST_POTENTIAL']*YIELD_RATE*3
zkp['ZKP_TOTAL_RETURN_3YR']    = zkp['ZKP_TAX_BENEFIT']+zkp['ZKP_YIELD_3YR']
zkp['CAPITAL_RETAINED']        = zkp['ZKP_INVEST_POTENTIAL']
zkp['REMITTANCE_OFFSET_PCT']   = (zkp['ZKP_INVEST_POTENTIAL']/
    zkp['EST_ANNUAL_REMITTANCE_MODEL'].replace(0,1)).clip(0,1)
 
# STEP 13: PUMA SUMMARY
puma = df.groupby(['GEOID_JOIN','STATE_NAME','COUNTRY_NAME']).agg(
    total_hh=('RPS','count'), avg_rps=('RPS','mean'),
    predicted_remitters=('REMITTER_PREDICTED','sum'),
    est_remittance_usd=('EST_ANNUAL_REMITTANCE_MODEL','sum'),
    avg_income=('INCWAGE','mean'), avg_housing_cost=('MONTHLY_HOUSING_COST','mean'),
    avg_yrsusa=('YRSUSA1','mean'),
    pct_non_citizen=('IS_CITIZEN',lambda x:(x==0).mean()),
    pct_recent_arrival=('RECENT_ARRIVAL','mean'),
).reset_index().round(4)
puma['remittance_M']  = (puma['est_remittance_usd']/1e6).round(3)
puma['remit_per_hh']  = (puma['est_remittance_usd']/puma['total_hh']).round(0)
puma['zkp_priority']  = ((puma['avg_rps']>=THRESHOLD)&(puma['pct_non_citizen']>=0.50)).astype(int)
print(f'ZKP priority PUMAs: {(puma["zkp_priority"]==1).sum()}')
 
# STEP 14: SAVE ALL (GEOID FIX APPLIED)
for frame in [df, puma, zkp]:
    frame['GEOID_JOIN'] = frame['GEOID_JOIN'].astype(str).str.zfill(7)
df.to_csv(OUTPUT_FILE, index=False, quoting=csv.QUOTE_NONNUMERIC)
puma.to_csv(PUMA_FILE, index=False, quoting=csv.QUOTE_NONNUMERIC)
zkp.to_csv(ZKP_FILE, index=False, quoting=csv.QUOTE_NONNUMERIC)
print()
print('='*60)
print('AI MODEL COMPLETE')
print('='*60)
print(f'V8:   {OUTPUT_FILE}  ({len(df):,} rows)')
print(f'PUMA: {PUMA_FILE}  ({len(puma):,} rows)')
print(f'ZKP:  {ZKP_FILE}  ({len(zkp):,} rows)')
print(f'AUC:  {roc:.4f}  |  ZKP targets: {len(zkp):,}')
print('Next: Run zkp_bond_simulation_FINAL.py')
print('='*60)


Loading data...
  351,893 records | GEOID: OK | IS_SHADOW: 15.27%
  Class: 0=298,162  1=53,731  spw=5.55
Building XGBoost + Calibration...
Running 5-fold CV (5-15 minutes)...

------------------------------------------------
5-FOLD CROSS-VALIDATION RESULTS
------------------------------------------------
  Recall      : 0.272 +/- 0.012
  Precision   : 0.575 +/- 0.005
  F1          : 0.369 +/- 0.011
  ROC_AUC     : 0.875 +/- 0.001
------------------------------------------------
Training final model...

------------------------------------------------
EVALUATION (threshold=0.45)
------------------------------------------------
              precision    recall  f1-score   support

Non-Remitter       0.90      0.94      0.92     59633
    Remitter       0.54      0.39      0.45     10746

    accuracy                           0.86     70379
   macro avg       0.72      0.66      0.68     70379
weighted avg       0.84      0.86      0.85     70379

ROC-AUC: 0.8757
Saved: remittance_featu